# Loan Repayment Prediction using Decision Tree  
## Instructor Answer Key Notebook

### Problem Statement
Design and develop a Machine Learning model that can help in **Loan Repayment Prediction** using the **Decision Tree** algorithm.

### Dataset Description
The Loan Repayment dataset contains **1000 rows** and **6 columns**:

- Initial payment  
- Last payment  
- Credit score  
- House number  
- Sum  
- Result  

The target variable is **Result**, which indicates whether the loan is repaid or not.

## Learning Goals
After completing this notebook, students will be able to:

1. Load and inspect a tabular dataset for classification.
2. Prepare features and target variables for model building.
3. Train a Decision Tree classifier.
4. Evaluate the model using accuracy, confusion matrix, and classification report.
5. Apply pruning to control overfitting.

## Step 1: Import Required Libraries
In this step, we import the Python libraries required for data handling, model building, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

## Step 2: Load the Dataset
This notebook first tries to load `loan_repayment.csv` from the current folder.

If the file is not available, it creates a **sample synthetic dataset** with the same structure so the notebook can still run end-to-end.

In [ ]:
from pathlib import Path

csv_path = Path("loan_repayment.csv")

if csv_path.exists():
    data = pd.read_csv(csv_path)
    print("Dataset loaded from file:", csv_path.resolve())
else:
    n = 1000
    initial_payment = np.random.randint(1000, 10000, n)
    last_payment = np.random.randint(500, 12000, n)
    credit_score = np.random.randint(300, 851, n)
    house_number = np.random.randint(0, 6, n)   # simplified numeric feature
    total_sum = initial_payment + last_payment

    # Create a target with realistic dependence on features
    risk_score = (
        0.004 * initial_payment
        + 0.003 * last_payment
        + 0.02 * credit_score
        + 2.5 * house_number
        - 0.0015 * total_sum
        + np.random.normal(0, 4, n)
    )
    result = (risk_score > np.median(risk_score)).astype(int)

    data = pd.DataFrame({
        "Initial payment": initial_payment,
        "Last payment": last_payment,
        "Credit score": credit_score,
        "House number": house_number,
        "Sum": total_sum,
        "Result": result
    })
    print("Sample dataset generated because loan_repayment.csv was not found.")

data.head()

## Step 3: Explore the Dataset
Now we inspect the shape, data types, summary statistics, and missing values.

In [ ]:
print("Shape of dataset:", data.shape)
print("\nData types:\n", data.dtypes)
print("\nMissing values:\n", data.isnull().sum())
print("\nSummary statistics:\n", data.describe())

## Step 4: Define Features and Target
We separate the independent variables (**X**) from the target variable (**y**).

In [ ]:
X = data.drop("Result", axis=1)
y = data["Result"]

print("Feature columns:", list(X.columns))
print("Target distribution:\n", y.value_counts())

## Step 5: Split the Dataset
We split the data into training and testing sets so the model can be trained on one portion and evaluated on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## Step 6: Build the Decision Tree Model
A Decision Tree classifier is created and trained using the training data.

Here, `max_depth=4` is used as an initial pruning choice to reduce overfitting.

In [ ]:
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)
print("Decision Tree model trained successfully.")

## Step 7: Make Predictions
The trained model predicts the class labels for the test set.

In [ ]:
y_pred = model.predict(X_test)
y_pred[:10]

## Step 8: Evaluate the Model
We evaluate the model using:

- Accuracy
- Confusion Matrix
- Classification Report

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

## Step 9: Interpret Feature Importance
Decision Trees provide feature importance values, which help us understand which features influence the predictions the most.

In [ ]:
feature_importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importance:\n")
print(feature_importance)

In [ ]:
feature_importance.plot(kind="bar")
plt.title("Feature Importance - Decision Tree")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

## Step 10: Control Overfitting Using Pruning
To study pruning, we compare multiple values of `max_depth`.

In [ ]:
depth_results = []

for depth in [2, 3, 4, 5, 6, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)

    train_pred = clf.predict(X_train)
    test_pred = clf.predict(X_test)

    depth_results.append({
        "max_depth": str(depth),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred)
    })

results_df = pd.DataFrame(depth_results)
results_df

In [ ]:
results_df.plot(x="max_depth", y=["train_accuracy", "test_accuracy"], marker="o")
plt.title("Effect of Tree Depth on Accuracy")
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

### Interpretation
- If training accuracy becomes very high while testing accuracy does not improve, the model may be overfitting.
- A moderate tree depth often gives better generalization.

## Step 11: Visualize the Decision Tree
This visualization helps students understand how the tree makes decisions.

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=["Not Repaid", "Repaid"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree for Loan Repayment Prediction")
plt.show()

## Step 12: Final Conclusion
The Decision Tree model can classify loan repayment outcomes using features such as payment values, credit score, and house number.

Pruning helps reduce overfitting and improves generalization. This is important because a model that performs well on training data alone may not perform well on new borrower data.

## Exercise Answers / Instructor Notes

**Q1. Change `max_depth` to different values and observe accuracy.**  
Expected observation: smaller depths may underfit, while very deep trees may overfit. Moderate depths often perform best.

**Q2. Which model performs better — full tree or pruned tree?**  
Typical answer: the pruned tree often performs better on test data because it generalizes better.

**Q3. Which feature is most important?**  
Answer depends on dataset values, but in many runs `Credit score` or payment-related variables become dominant.

**Q4. What happens when the tree is too deep?**  
It may memorize training data and suffer from overfitting.

## Optional Extension
Students can try:

- `criterion="entropy"` instead of `"gini"`
- `min_samples_split`
- `min_samples_leaf`
- cost-complexity pruning with `ccp_alpha`